<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part A: Foundations and Data Exploration</h2>
<h2>Notebook A04: Handling Outliers</h2>
</div>

An outlier is an observation that does not fit the pattern of the rest of the series. Some are
recording errors you want to remove. Others are the most interesting thing in the dataset: a stock-out,
a heatwave, a system failure. Deciding which is which is a judgement call, and it is one you cannot make
without first understanding the structure of your data.

That is the theme of this notebook. Most of the work in outlier detection is not running a detector, it
is preparing the series so that the detector is answering the right question.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [What Counts as an Outlier](#2.-What-Counts-as-an-Outlier)
3. [Structure Before Statistics](#3.-Structure-Before-Statistics)
4. [Global Methods: Standard Deviation and IQR](#4.-Global-Methods:-Standard-Deviation-and-IQR)
5. [Removing the Pattern First](#5.-Removing-the-Pattern-First)
6. [A Formal Test: Generalized ESD](#6.-A-Formal-Test:-Generalized-ESD)
7. [Isolation Forest](#7.-Isolation-Forest)
8. [What to Do With the Outliers You Found](#8.-What-to-Do-With-the-Outliers-You-Found)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest

import nb_config

sns.set_theme(style="whitegrid")

We use the Rossmann store sales dataset: daily sales for 1,115 drugstores in Germany. If you have not
prepared it yet, run Notebook [F01c](./F01c_Preparing_external_datasets.ipynb) first.

The file holds every store in one long table, so we start by picking a single store. Store 1 is a typical
one, and working with a single series keeps the code readable. Everything in this notebook applies
unchanged to any other store.

In [ ]:
sales = pd.read_csv(
    nb_config.ROSSMANN_TRAIN_PATH,
    parse_dates=["Date"],
    low_memory=False,
)

# One store, indexed by date
store = (
    sales[sales["Store"] == 1]
    .set_index("Date")
    .sort_index()
    .loc[:, ["Sales", "Customers", "Open", "Promo", "StateHoliday", "SchoolHoliday"]]
)

print(f"Rows:  {len(store)}")
print(f"Range: {store.index.min().date()} to {store.index.max().date()}")
store.head()

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-What-Counts-as-an-Outlier">2. What Counts as an Outlier</h3>
</div>

It helps to separate three things that all look like "an unusual value" on a plot:

- **Global outliers** are extreme compared to the series as a whole. The hottest day in a century.
- **Contextual outliers** are only extreme *given their context*. 15 °C is unremarkable in April and very
  strange in January. These are the ones that matter most in time series, and the ones that simple
  statistical rules miss.
- **Structural values** are not outliers at all. They are a documented part of how the data is generated:
  a shop that is closed, a sensor that writes a fixed code when it fails.

Start, as always, by looking at the series.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(store["Sales"], color="steelblue", linewidth=0.7)

ax.set_title("Daily sales, store 1", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The plot is dominated by a thick band of values dropping to zero. Before reading anything else into the
series, we need to know what those zeros are.

In [ ]:
zero_days = store[store["Sales"] == 0]

print(f"Days with zero sales: {len(zero_days)} of {len(store)}")
print(f"Of those, closed:     {(zero_days['Open'] == 0).sum()}")
print()
print("Weekday distribution of zero-sales days (0 = Monday):")
print(zero_days.index.dayofweek.value_counts().sort_index())

Every zero is a day the store was closed, and almost all of them are Sundays. These are structural
values, not anomalies. Leaving them in would not just add noise, it would actively break the detectors
we are about to use, as the next section shows.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Structure-Before-Statistics">3. Structure Before Statistics</h3>
</div>

The most common outlier rule is to flag anything more than three standard deviations from the mean. Let
us apply it to the raw series and see what happens.

In [ ]:
def sigma_bounds(series, n_sigma=3):
    """Lower and upper bound of the n-sigma rule."""
    mean, std = series.mean(), series.std()
    return mean - n_sigma * std, mean + n_sigma * std


raw = store["Sales"]
open_days = store[store["Open"] == 1]["Sales"]

for label, series in [("All days", raw), ("Open days only", open_days)]:
    low, high = sigma_bounds(series)
    flagged = ((series < low) | (series > high)).sum()
    print(f"{label:<16} mean={series.mean():>7.0f}  std={series.std():>7.0f}  "
          f"bounds=({low:>7.0f}, {high:>7.0f})  flagged={flagged}")

On the raw series the rule flags **nothing at all**. The 161 closed days pull the mean down and roughly
double the standard deviation, which widens the bounds so far that no real day can ever fall outside
them. The structural zeros have hidden every anomaly in the series.

Dropping the closed days halves the standard deviation and the same rule now flags ten days. This is the
single most important habit in outlier work: **understand and remove structure before you measure
spread**. No detector can do this step for you, because only you know that `Open == 0` means the shop
was shut.

In [ ]:
# From here on we work only with days the store was actually open
opened = store[store["Open"] == 1].copy()

print(f"Open days: {len(opened)} of {len(store)}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Global-Methods:-Standard-Deviation-and-IQR">4. Global Methods: Standard Deviation and IQR</h3>
</div>

Two classical rules, both applied to the distribution of values without any reference to time:

- **Standard deviation.** Flag points more than *n* standard deviations from the mean, usually n = 3.
  Assumes roughly normal data, and is sensitive to the very outliers it is looking for: a few extreme
  points inflate the standard deviation and hide themselves.
- **Interquartile range (IQR).** Flag points outside `[Q1 - 1.5 x IQR, Q3 + 1.5 x IQR]`. Quartiles are
  barely affected by a handful of extreme values, so this is the more robust of the two, and it makes no
  assumption about the shape of the distribution.

In [ ]:
def iqr_bounds(series, k=1.5):
    """Lower and upper bound of the Tukey IQR rule."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


sales_open = opened["Sales"]

methods = {
    "3-sigma": sigma_bounds(sales_open),
    "IQR": iqr_bounds(sales_open),
}

for name, (low, high) in methods.items():
    mask = (sales_open < low) | (sales_open > high)
    opened[f"outlier_{name}"] = mask
    print(f"{name:<9} bounds=({low:>7.0f}, {high:>7.0f})  flagged={mask.sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(sales_open, color="steelblue", linewidth=0.7, label="Sales (open days)")

for name, colour, marker in [("3-sigma", "crimson", "o"), ("IQR", "darkorange", "x")]:
    flagged = sales_open[opened[f"outlier_{name}"]]
    ax.scatter(flagged.index, flagged, color=colour, marker=marker, s=45,
               label=f"{name} ({len(flagged)})", zorder=3)

low, high = methods["IQR"]
ax.axhline(low, color="darkorange", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axhline(high, color="darkorange", linestyle="--", linewidth=0.8, alpha=0.7)

ax.set_title("Global outlier rules on open days", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Now look at *when* the flagged days fall.

In [ ]:
flagged_any = opened[opened["outlier_3-sigma"] | opened["outlier_IQR"]]

print("Month of flagged days:")
print(flagged_any.index.month.value_counts().sort_index().to_string())

Almost every flagged day is in December. That is not a detector finding anomalies, it is a detector
rediscovering Christmas.

Both rules compare each day against the average of the *whole* series. December is genuinely busier than
the rest of the year, so December days look extreme by that comparison even though they are entirely
normal for December. These are false positives created by seasonality, and they are exactly the
contextual outliers we defined in section 2, seen from the wrong angle.

**Exercise.** Apply the IQR rule separately within each calendar month (group by `opened.index.month` and compute the bounds per group). How many days does it flag now, and in which months?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Removing-the-Pattern-First">5. Removing the Pattern First</h3>
</div>

The fix is to stop comparing each day against the whole series, and start comparing it against what we
*expect* for that kind of day. Anything left over after removing the expected pattern is a residual, and
a residual is what we should be running the detector on.

Sales here follow two patterns we can see plainly: a weekly one (Mondays are busiest, Sundays are
closed) and a yearly one (December is busy). So we build an expected value for every day from the median
of days sharing the same month and weekday, and subtract it.

Using the median rather than the mean matters: the median is not dragged around by the very outliers we
are trying to find.

In [ ]:
opened["month"] = opened.index.month
opened["weekday"] = opened.index.dayofweek

# Expected sales for a day, given its month and weekday
opened["expected"] = opened.groupby(["month", "weekday"])["Sales"].transform("median")
opened["residual"] = opened["Sales"] - opened["expected"]

print(f"Standard deviation of raw sales: {opened['Sales'].std():.0f}")
print(f"Standard deviation of residuals: {opened['residual'].std():.0f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(opened["Sales"], color="steelblue", linewidth=0.7, label="Actual")
axes[0].plot(opened["expected"], color="darkorange", linewidth=1.0, label="Expected (month x weekday median)")
axes[0].set_title("Sales and the expected seasonal pattern", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Sales")
axes[0].legend(loc="upper left")

axes[1].plot(opened["residual"], color="seagreen", linewidth=0.7)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("Residuals: what the seasonal pattern does not explain", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Residual")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The residual series is centred on zero and has no visible seasonal shape left. Now the IQR rule is
asking a sensible question: which days differ from what we expected *for that day*?

In [ ]:
low, high = iqr_bounds(opened["residual"])
opened["outlier_residual"] = (opened["residual"] < low) | (opened["residual"] > high)

print(f"IQR on residuals  bounds=({low:.0f}, {high:.0f})  flagged={opened['outlier_residual'].sum()}")
print()
print("Month of flagged days:")
print(opened[opened["outlier_residual"]].index.month.value_counts().sort_index().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(opened["residual"], color="seagreen", linewidth=0.7, label="Residual")

flagged = opened.loc[opened["outlier_residual"], "residual"]
ax.scatter(flagged.index, flagged, color="crimson", s=45, zorder=3,
           label=f"Flagged ({len(flagged)})")

ax.axhline(low, color="crimson", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axhline(high, color="crimson", linestyle="--", linewidth=0.8, alpha=0.7)
ax.axhline(0, color="gray", linewidth=0.8)

ax.set_title("Outliers found on the residuals", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Residual")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# The largest deviations, in both directions
flagged_days = opened[opened["outlier_residual"]].copy()

print("Ten biggest shortfalls:")
print(flagged_days.nsmallest(10, "residual")[["Sales", "expected", "residual", "Promo"]].round(0).to_string())

December still dominates the list, but the picture underneath has changed. Ten of the sixteen days the
global rules flagged are no longer unusual once December's own level is taken into account: they were
busy, but no busier than December predicts.

What rises to the top instead are the two largest shortfalls, **24 and 31 December**, when the store
closed early and sold far *less* than a normal December day. Those are genuine anomalies, and the global
rules missed both of them because a below-average December day still looks perfectly ordinary next to the
yearly average.

This is the shape of nearly every real outlier analysis: model the structure you understand, then look at
what is left.

**Exercise.** The `Promo` column marks promotion days, which lift sales by around 1,000 on average. Add it to the grouping (`['month', 'weekday', 'Promo']`) and recompute the residuals and flags. Does accounting for promotions remove any of the days flagged above?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-A-Formal-Test:-Generalized-ESD">6. A Formal Test: Generalized ESD</h3>
</div>

The IQR rule gives you a cut-off, but not a probability: it cannot say how surprised you should be. The
**Generalized Extreme Studentized Deviate** test does. It repeatedly takes the point furthest from the
mean, compares it against a critical value from the t-distribution, removes it, and repeats up to a
maximum you choose. The result is the number of outliers the data supports at a given significance level.

Applying ESD to *deseasonalised residuals* rather than raw values is exactly what the literature calls
**S-ESD** (Seasonal ESD). We already have the residuals, so we get S-ESD for free.

In [ ]:
def generalized_esd(values, max_outliers=20, alpha=0.05):
    """Generalized ESD test. Returns the positions of the detected outliers.

    Tests the `max_outliers` most extreme points in turn and reports how many of
    them are significant at level `alpha`.
    """
    values = np.asarray(values, dtype=float)
    n = len(values)

    working = values.copy()
    positions = np.arange(n)
    test_stats, critical_values, candidates = [], [], []

    for i in range(1, max_outliers + 1):
        std = working.std(ddof=1)
        if std == 0:
            break

        deviations = np.abs(working - working.mean()) / std
        worst = deviations.argmax()

        test_stats.append(deviations[worst])
        candidates.append(positions[worst])

        # Critical value for this iteration
        size = n - i + 1
        t_crit = stats.t.ppf(1 - alpha / (2 * size), size - 2)
        critical_values.append(
            (size - 1) * t_crit / np.sqrt((size - 2 + t_crit**2) * size)
        )

        working = np.delete(working, worst)
        positions = np.delete(positions, worst)

    # The number of outliers is the largest i for which the test statistic exceeds
    # its critical value
    n_outliers = 0
    for i, (statistic, critical) in enumerate(zip(test_stats, critical_values), start=1):
        if statistic > critical:
            n_outliers = i

    return np.sort(np.array(candidates[:n_outliers], dtype=int))

In [ ]:
esd_positions = generalized_esd(opened["residual"].values, max_outliers=30, alpha=0.05)

opened["outlier_esd"] = False
opened.iloc[esd_positions, opened.columns.get_loc("outlier_esd")] = True

print(f"S-ESD flagged {len(esd_positions)} day(s) at alpha = 0.05")
print()
print(opened[opened["outlier_esd"]][["Sales", "expected", "residual"]].round(0).to_string())

S-ESD is far more conservative than the IQR rule: one day instead of nineteen. That is not a bug, it is
what a hypothesis test is for. The IQR rule flags everything beyond a fixed cut-off; ESD only flags what
is unlikely enough to survive a formal test, and it corrects for the fact that you are testing many
points at once.

Which you want depends on the cost of being wrong. Screening data before modelling, take the generous
list and inspect it. Raising alerts that a human has to act on, take the strict one. `alpha` and
`max_outliers` are the dials in between.

**Exercise.** Re-run the ESD test with `alpha=0.20`. How many days does it flag, and how does the list compare with the days the IQR rule found?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Isolation-Forest">7. Isolation Forest</h3>
</div>

Everything so far has looked at one number at a time. **Isolation Forest** takes several columns at once
and asks a different question: how easy is this point to separate from the rest?

It builds random trees that split the data on random features at random thresholds. Points that end up
isolated after only a few splits are, by construction, unusual. It assumes nothing about the
distribution, handles several variables at once, and needs no labelled examples.

The one parameter that matters in practice is `contamination`, the fraction of the data you expect to be
anomalous. It is not learned from the data, you assert it, so treat it as the dial it is.

In [ ]:
features = opened[["Sales", "residual", "Promo", "weekday"]]

forest = IsolationForest(contamination=0.02, random_state=42)
opened["outlier_forest"] = forest.fit_predict(features) == -1

# The lower the score, the more anomalous the point
opened["forest_score"] = forest.score_samples(features)

print(f"Isolation Forest flagged {opened['outlier_forest'].sum()} days "
      f"({opened['outlier_forest'].mean():.1%} of open days)")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

scatter = ax.scatter(
    opened.index,
    opened["Sales"],
    c=opened["forest_score"],
    cmap="viridis",
    s=12,
)

flagged = opened.loc[opened["outlier_forest"], "Sales"]
ax.scatter(flagged.index, flagged, facecolors="none", edgecolors="crimson",
           s=90, linewidths=1.5, label=f"Flagged ({len(flagged)})", zorder=3)

ax.set_title("Isolation Forest: anomaly score and flagged days", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.colorbar(scatter, ax=ax, label="Anomaly score (lower = more anomalous)")

plt.tight_layout()
plt.show()

How much do the three approaches actually agree?

In [ ]:
comparison = pd.DataFrame({
    "IQR on residuals": opened["outlier_residual"],
    "S-ESD": opened["outlier_esd"],
    "Isolation Forest": opened["outlier_forest"],
})

print("Days flagged by each method:")
print(comparison.sum().to_string())
print()
print("Days flagged by how many methods:")
print(comparison.sum(axis=1).value_counts().sort_index().to_string())
print()
print("Flagged by all three:")
print(opened.loc[comparison.all(axis=1), ["Sales", "expected", "residual"]].round(0).to_string())

The methods overlap but do not agree, and no amount of tuning will make them. Each encodes a different
idea of "unusual": a fixed distance from the middle, a statistically improbable deviation, or a point
that is easy to isolate in several dimensions at once.

Treat the days that every method flags as your high-confidence findings, and the rest as a list worth
looking at rather than a verdict.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-What-to-Do-With-the-Outliers-You-Found">8. What to Do With the Outliers You Found</h3>
</div>

Detection is the easy half. Now you have to decide what happens to the points you flagged, and that is a
question about your problem, not about statistics. Four options, in rough order of how often they are the
right one:

**Keep them, and record them.** The default. Add a boolean column and carry on. Most forecasting models
cope with a handful of unusual points, and you keep the option of explaining them later.

**Explain them with a feature.** The best outcome. If the flagged days are Christmas Eve, a `holiday`
column turns an anomaly into information the model can use. Nothing gets discarded, and the model gets
better.

**Replace them.** Treat the value as missing and impute it with the methods from Notebook
[A03](./A03_Handling_missing_data.ipynb). Reasonable when you know the value is *wrong*, for example a
sensor fault. Do not do it merely because a value is inconvenient.

**Remove them.** Rarely right for time series: dropping a row leaves a hole in the index, and most models
assume regular spacing. Prefer replacing.

Whatever you choose, do it after the train/test split and never using information from the future. The
cell below applies the two most common options.

In [ ]:
treated = opened.copy()

# Option 1: keep the values, record the finding
treated["is_outlier"] = treated["outlier_residual"]

# Option 2: replace flagged values with the seasonal expectation, then interpolate
treated["sales_cleaned"] = treated["Sales"].where(~treated["is_outlier"])
treated["sales_cleaned"] = treated["sales_cleaned"].interpolate(method="time")

changed = treated.loc[treated["is_outlier"], ["Sales", "expected", "sales_cleaned"]]
print(f"Replaced {len(changed)} values:")
print(changed.round(0).head(8).to_string())

In [ ]:
window = slice("2013-12-01", "2014-01-15")

fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(treated.loc[window, "Sales"], color="steelblue", linewidth=1.2,
        marker="o", markersize=4, label="Original")
ax.plot(treated.loc[window, "sales_cleaned"], color="darkorange", linewidth=1.2,
        linestyle="--", marker="s", markersize=4, label="After replacement")

flagged = treated.loc[window].query("is_outlier")
ax.scatter(flagged.index, flagged["Sales"], color="crimson", s=70, zorder=3, label="Flagged")

ax.set_title("Christmas 2013: original and cleaned series", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Sales")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Seeing it up close makes the trade-off concrete. Replacing the 24 and 31 December values removes two
sharp dips, and with them the information that this shop closes early on those days. If your goal is a
model that forecasts next Christmas, that information is exactly what you want to keep, as a holiday
feature rather than a hole.

Cleaning a series is not automatically an improvement. Ask what the flagged point means before you
decide.

**Exercise.** Load the Air Quality dataset (`nb_config.AIR_QUALITY_PATH`, with `sep=';'` and `decimal=','`). Missing readings in that file are coded as `-200` rather than left empty. Find them, replace them with `NaN`, and then run the residual-based detection from section 5 on the `C6H6(GT)` column using hour-of-day as the seasonal grouping. What would have happened if you had left the -200 values in?

In [ ]:
# Your solution here


---

Outliers are the last of the data-quality problems we cover. You can now load a series, visualise it,
deal with gaps, and decide what to do about values that do not fit.

The next notebook starts the forecasting half of the course. Before reaching for a sophisticated model,
we build the simple baselines that any serious model has to beat:
[A05 - Forecasting Baselines](./A05_Forecasting_baselines.ipynb).

**Solutions.** Worked answers to the 4 exercises above, with the reasoning behind them, are in
[A04_Handling_outliers_solutions.ipynb](../solutions/A04_Handling_outliers_solutions.ipynb). Try each one yourself first.
